# Beijing Virtual Air-Quality Stations: solution v3 (final)

Same features as v2. It trains two complementary LightGBM models per pollutant, one on the log concentration and one on the station's offset from the leave-self-out network mean. It blends them with weights chosen on leave-one-station-out predictions and reports per-station diagnostics. Things tried during development that did **not** help under LOSO (and so are left out): nearest-neighbour readings picked by weather similarity, proximity-weighted network means, station-level climate descriptors, and heavier regularisation.

## Setup
Paths follow the platform convention: read from `./dataset/public/`, write to `./working/`.

In [1]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore", category=UserWarning)
T0 = time.time()
DATA = "./dataset/public/"
OUT = "./working/"
os.makedirs(OUT, exist_ok=True)

POL = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
MET = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
DIRS = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
        "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]

train = pd.read_csv(DATA + "train.csv")
test = pd.read_csv(DATA + "test.csv")
for d in (train, test):
    d["ts"] = pd.to_datetime(d[["year", "month", "day", "hour"]])

NET = sorted(train.station.unique())   # 8 labelled network stations
HELD = sorted(test.station.unique())   # 4 stations to reconstruct
print("train", train.shape, "stations:", NET)
print("test ", test.shape, "stations:", HELD)

train (280512, 19) stations: ['Aotizhongxin', 'Changping', 'Dingling', 'Dongsi', 'Guanyuan', 'Shunyi', 'Tiantan', 'Wanshouxigong']
test  (140256, 13) stations: ['Gucheng', 'Huairou', 'Nongzhanguan', 'Wanliu']


## 1. Quick look at the data
The pollutant columns are right-skewed and have multi-hour gaps. The metric is RMSE in
log1p space, so everything below is modelled on `log1p(concentration)`.

In [2]:
print(train[POL].describe().T[["mean", "50%", "max"]])
print("\nMissing fraction per pollutant (train):")
print(train[POL].isna().mean().round(4))
print("\nMissing fraction per column (test):")
print(test[MET + ["wd"]].isna().mean().round(4))

              mean    50%      max
PM2.5    79.462429   55.0    999.0
PM10    103.151091   81.0    999.0
SO2      15.669286    7.0    411.0
NO2      49.438548   42.0    290.0
CO     1222.307078  900.0  10000.0
O3       57.908371   46.0   1071.0

Missing fraction per pollutant (train):
PM2.5    0.0219
PM10     0.0163
SO2      0.0232
NO2      0.0287
CO       0.0530
O3       0.0313
dtype: float64

Missing fraction per column (test):
TEMP    0.0010
PRES    0.0010
DEWP    0.0010
RAIN    0.0010
WSPM    0.0008
wd      0.0047
dtype: float64


## 2. Wide hourly matrices
One matrix per pollutant: rows = hours, columns = network stations, values = log1p
concentration. Missing readings stay NaN; they are never filled with 0.

In [3]:
TIMES = pd.DatetimeIndex(sorted(train.ts.unique()))
LOGW = {p: np.log1p(train.pivot(index="ts", columns="station", values=p).reindex(TIMES))
        for p in POL}
allrows = pd.concat([train, test], ignore_index=True)
# Weather matrices for all 12 sites (weather is known everywhere, including test sites).
METW = {m: allrows.pivot(index="ts", columns="station", values=m).reindex(TIMES) for m in MET}

## 3. Baseline under leave-one-station-out (LOSO) validation
The test set is four *unseen* stations, so validation must hold out whole stations.
A random row split would leak each station's level through autocorrelated neighbouring
hours. Baseline: for station s, predict the hourly log-mean of the *other* network stations.

In [4]:
def rmsle_log(pred_log, true_log):
    m = ~np.isnan(true_log) & ~np.isnan(pred_log)
    return float(np.sqrt(np.mean((pred_log[m] - true_log[m]) ** 2)))

base_scores = {}
for p in POL:
    errs = []
    for s in NET:
        others = [o for o in NET if o != s]
        pred = LOGW[p][others].mean(axis=1).to_numpy()
        errs.append(rmsle_log(pred, LOGW[p][s].to_numpy()))
    base_scores[p] = np.mean(errs)
print("LOSO network-mean baseline:", {k: round(v, 4) for k, v in base_scores.items()})
print("LOSO network-mean MCRMSLE: %.4f" % np.mean(list(base_scores.values())))

LOSO network-mean baseline: {'PM2.5': np.float64(0.4094), 'PM10': np.float64(0.4436), 'SO2': np.float64(0.4748), 'NO2': np.float64(0.5093), 'CO': np.float64(0.4114), 'O3': np.float64(0.6342)}
LOSO network-mean MCRMSLE: 0.4804


## 4. Features (same as v2)
Everything from v1:
* calendar features
* local weather with circular wind encoding
* leave-self-out network aggregates

plus **temporal context**. This is reconstruction, not forecasting, so hours before *and*
after the target hour are fair game:
* centred rolling means (3, 6 and 24 h) of the network mean, which smooth sensor noise
  and capture regional episodes;
* one-hour differences backwards and forwards, which capture plume arrival lag between
  sites;
* local weather relative to the network average (TEMP, PRES, DEWP, WSPM). A lower pressure
  than the network means a higher, more suburban or hill-side site. Those sites see less
  traffic NO2 and more O3, the dominant error source for sites outside the urban core;
* 24 h summaries of local wind (speed, u, v) and rain, plus the temperature anomaly
  against its 24 h mean (a stability / inversion proxy);
* network Ox = NO2 + O3. Ozone titration by NO makes Ox much smoother in space than either
  gas alone, which helps the O3 and NO2 targets.

In [5]:
WD_ANGLE = {d: i * np.pi / 8 for i, d in enumerate(DIRS)}

def station_features(s):
    others = [o for o in NET if o != s]
    g = allrows[allrows.station == s].set_index("ts").reindex(TIMES)
    f = {"id": g["id"].to_numpy(), "station": np.full(len(TIMES), s)}
    f["hour"] = TIMES.hour; f["month"] = TIMES.month
    f["doy"] = TIMES.dayofyear; f["dow"] = TIMES.dayofweek
    f["t"] = (TIMES - TIMES[0]).days.to_numpy()
    for m in MET:
        f[m] = g[m].to_numpy()
    ang = g["wd"].map(WD_ANGLE).to_numpy(dtype=float)
    f["wd_sin"], f["wd_cos"] = np.sin(ang), np.cos(ang)
    f["u"], f["v"] = -g["WSPM"].to_numpy() * np.sin(ang), -g["WSPM"].to_numpy() * np.cos(ang)
    f["dew_dep"] = f["TEMP"] - f["DEWP"]
    for m in ("TEMP", "PRES", "DEWP", "WSPM"):
        # local weather relative to the network: pressure offset ~ elevation, temperature /
        # wind offsets ~ urban heat island and exposure; i.e. "how suburban is this site"
        f[f"{m}_vs_net"] = f[m] - METW[m][others].mean(axis=1).to_numpy()
    for p in POL:
        w = LOGW[p][others]
        net_mean = w.mean(axis=1)
        f[f"net_mean_{p}"] = net_mean.to_numpy()
        f[f"net_std_{p}"] = w.std(axis=1).to_numpy()
        f[f"net_min_{p}"] = w.min(axis=1).to_numpy()
        f[f"net_max_{p}"] = w.max(axis=1).to_numpy()
        for k in (3, 6, 24):
            f[f"net_mean_{p}_r{k}"] = net_mean.rolling(k, center=True, min_periods=1).mean().to_numpy()
        f[f"net_mean_{p}_d1"] = net_mean.diff().to_numpy()
        f[f"net_mean_{p}_l1"] = net_mean.diff(-1).to_numpy()
        f[f"y_{p}"] = LOGW[p][s].to_numpy() if s in NET else np.full(len(TIMES), np.nan)
    f["net_Ox"] = np.log1p(np.expm1(f["net_mean_NO2"]) + np.expm1(f["net_mean_O3"]))
    local = pd.DataFrame({"WSPM": f["WSPM"], "u": f["u"], "v": f["v"],
                          "RAIN": f["RAIN"], "TEMP": f["TEMP"]})
    for c in ("WSPM", "u", "v"):
        f[f"{c}_r24"] = local[c].rolling(24, center=True, min_periods=1).mean().to_numpy()
    f["rain_r24"] = local["RAIN"].rolling(24, center=True, min_periods=1).sum().to_numpy()
    f["temp_anom24"] = (local["TEMP"] - local["TEMP"].rolling(24, center=True, min_periods=1).mean()).to_numpy()
    return pd.DataFrame(f)

FEAT = pd.concat([station_features(s) for s in NET + HELD], ignore_index=True)
COLS = [c for c in FEAT.columns if c not in ("id", "station") and not c.startswith("y_")]
TR = FEAT[FEAT.station.isin(NET)].reset_index(drop=True)
TE = FEAT[FEAT.station.isin(HELD)].reset_index(drop=True)
print("feature matrix:", TR.shape, TE.shape, "| features:", len(COLS))

feature matrix: (280512, 87) (140256, 87) | features: 79


## 5. Two complementary LightGBM models per pollutant (LOSO folds)
* **direct**: predicts `log1p(y)`.
* **residual**: predicts `log1p(y) - net_mean`, the station's offset from the network
  (leave-self-out) mean, which is added back afterwards. This target is stationary
  across pollution episodes, so the trees spend their splits on *local* deviations
  (traffic, terrain, distance from the urban core) instead of re-learning the regional
  level.

Both use the same 8 leave-one-station-out folds. Out-of-fold predictions are kept so the
blend weight can be chosen honestly. Rows with a missing label are dropped per pollutant.

In [6]:
# Changed in v2: smaller trees, larger leaves and stronger L2. With only 8 training stations,
# deep trees can memorise station-specific quirks. LOSO is within noise of v1's settings,
# but this version trains about 2.5x faster, which leaves time budget for the v3 ensemble.
PARAMS = dict(objective="regression", learning_rate=0.08, num_leaves=31, min_data_in_leaf=200,
              feature_fraction=0.6, bagging_fraction=0.7, bagging_freq=1, lambda_l2=5.0,
              verbose=-1, num_threads=os.cpu_count(), seed=0)

def fit_family(p, residual, rounds=1500):
    y = TR[f"y_{p}"].to_numpy()
    fill = np.nanmean(TR[f"net_mean_{p}"])
    off_tr = TR[f"net_mean_{p}"].fillna(fill).to_numpy() if residual else 0.0
    off_te = TE[f"net_mean_{p}"].fillna(fill).to_numpy() if residual else 0.0
    t = y - off_tr
    ok = ~np.isnan(t)
    oof = np.full(len(TR), np.nan)
    pt = np.zeros(len(TE))
    for s in NET:
        va = (TR.station == s).to_numpy()
        dtr = lgb.Dataset(TR.loc[ok & ~va, COLS], t[ok & ~va])
        dva = lgb.Dataset(TR.loc[ok & va, COLS], t[ok & va])
        m = lgb.train(PARAMS, dtr, rounds, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[ok & va] = m.predict(TR.loc[ok & va, COLS], num_iteration=m.best_iteration)
        pt += m.predict(TE[COLS], num_iteration=m.best_iteration) / len(NET)
    return oof + off_tr, pt + off_te

OOF, PRED = {}, {}
for p in POL:
    for fam in ("direct", "residual"):
        OOF[p, fam], PRED[p, fam] = fit_family(p, residual=(fam == "residual"))
    y = TR[f"y_{p}"].to_numpy()
    print(f"{p:6s} LOSO direct {rmsle_log(np.clip(OOF[p, 'direct'], 0, None), y):.4f} | "
          f"residual {rmsle_log(np.clip(OOF[p, 'residual'], 0, None), y):.4f} | "
          f"baseline {base_scores[p]:.4f}  [{time.time() - T0:.0f}s]")

PM2.5  LOSO direct 0.3753 | residual 0.3757 | baseline 0.4094  [179s]


PM10   LOSO direct 0.4191 | residual 0.4174 | baseline 0.4436  [265s]


SO2    LOSO direct 0.4796 | residual 0.4707 | baseline 0.4748  [339s]


NO2    LOSO direct 0.5241 | residual 0.5066 | baseline 0.5093  [428s]


CO     LOSO direct 0.4026 | residual 0.3967 | baseline 0.4114  [602s]


O3     LOSO direct 0.5895 | residual 0.5853 | baseline 0.6342  [685s]


## 6. Blend weight per pollutant, chosen on LOSO out-of-fold predictions

In [7]:
WEIGHTS, cv3, pred_log = {}, {}, {}
for p in POL:
    y = TR[f"y_{p}"].to_numpy()
    grid = np.linspace(0, 1, 11)
    errs = [rmsle_log(np.clip(w * OOF[p, "direct"] + (1 - w) * OOF[p, "residual"], 0, None), y)
            for w in grid]
    w = float(grid[int(np.argmin(errs))])
    WEIGHTS[p], cv3[p] = w, min(errs)
    pred_log[p] = np.clip(w * PRED[p, "direct"] + (1 - w) * PRED[p, "residual"], 0, None)
print("direct-model weight per pollutant:", WEIGHTS)
print("LOSO per pollutant:", {k: round(v, 4) for k, v in cv3.items()})
print("LOSO MCRMSLE (blend): %.4f  vs network-mean baseline %.4f"
      % (np.mean(list(cv3.values())), np.mean(list(base_scores.values()))))

direct-model weight per pollutant: {'PM2.5': 0.5, 'PM10': 0.4, 'SO2': 0.30000000000000004, 'NO2': 0.2, 'CO': 0.30000000000000004, 'O3': 0.4}
LOSO per pollutant: {'PM2.5': 0.3735, 'PM10': 0.4158, 'SO2': 0.4689, 'NO2': 0.5045, 'CO': 0.3943, 'O3': 0.5802}
LOSO MCRMSLE (blend): 0.4562  vs network-mean baseline 0.4804


## 7. Per-station LOSO diagnostics
The suburban/rural network sites (Changping, Dingling, Shunyi) are the hardest to
reconstruct, and NO2/O3 are the hardest pollutants. They carry the largest local offsets
from the urban network. Expect the held-out suburban site (Huairou) to behave the same.

In [8]:
rows = {}
for s in NET:
    k = (TR.station == s).to_numpy()
    rows[s] = {p: rmsle_log(np.clip(WEIGHTS[p] * OOF[p, "direct"][k]
                                    + (1 - WEIGHTS[p]) * OOF[p, "residual"][k], 0, None),
                            TR[f"y_{p}"].to_numpy()[k]) for p in POL}
diag = pd.DataFrame(rows).T
diag["mean"] = diag.mean(axis=1)
print(diag.round(3).sort_values("mean"))

               PM2.5   PM10    SO2    NO2     CO     O3   mean
Guanyuan       0.269  0.319  0.366  0.288  0.309  0.518  0.345
Wanshouxigong  0.355  0.410  0.342  0.296  0.321  0.453  0.363
Dongsi         0.312  0.365  0.473  0.276  0.300  0.479  0.367
Aotizhongxin   0.284  0.344  0.403  0.336  0.308  0.560  0.372
Tiantan        0.307  0.359  0.475  0.361  0.276  0.521  0.383
Changping      0.440  0.439  0.475  0.459  0.423  0.610  0.475
Shunyi         0.442  0.485  0.621  0.538  0.502  0.721  0.551
Dingling       0.508  0.552  0.537  1.026  0.596  0.718  0.656


## 8. Write the submission

In [9]:
sub = pd.DataFrame({"id": TE["id"]})
for p in POL:
    sub[p] = np.expm1(pred_log[p]).clip(min=0)
assert len(sub) == len(test) and set(sub.id) == set(test.id)
assert np.isfinite(sub[POL].to_numpy()).all()
sub.to_csv(OUT + "submission.csv", index=False)
print(sub.head())
print("saved", OUT + "submission.csv", f"total time {time.time() - T0:.0f}s")

                   id     PM2.5      PM10       SO2        NO2          CO  \
0  Gucheng_2013030100  5.621316  7.397309  5.653105   9.702631  279.418004   
1  Gucheng_2013030101  6.253062  7.722663  5.472462   9.468813  280.447159   
2  Gucheng_2013030102  6.445034  7.761505  6.838316  12.796558  307.395120   
3  Gucheng_2013030103  6.366524  7.297883  6.037355  11.829765  312.882103   
4  Gucheng_2013030104  6.009210  6.713325  6.359156   9.258495  295.999389   

          O3  
0  67.841799  
1  68.334017  
2  59.192938  
3  67.090383  
4  60.116235  
saved ./working/submission.csv total time 687s
